# **Hito 2 · Sistema RAG**

**Procesamiento del Lenguaje Natural (UC3M, 2026/27)** · Chatbot sobre documentación de misiones espaciales

Este notebook continúa `hito1.ipynb`. Parte del **mismo corpus**, los **mismos fragmentos** (la unidad de indexación no cambia, para que la comparación con el Hito 1 sea justa) y las **mismas consultas** de `queries.jsonl`.

El Hito 2 se construye paso a paso:

1. ✅ **Sentence embeddings** de los fragmentos
2. ✅ **Base de datos vectorial** (Chroma)
3. ⬜ Recuperación densa y evaluación con las consultas del Hito 1 (Recall@5, Recall@10, MRR@10), comparada con TF-IDF y BM25
4. ⬜ Generación con un LLM de pesos abiertos (citar fuentes, no responder sin información, mismo idioma que la pregunta)
5. ⬜ Primera evaluación de la generación

## **0. Configuración**

El notebook se ejecuta desde la raíz del repositorio, como `hito1.ipynb`. En **Google Colab** (recomendable con GPU: *Entorno de ejecución → Cambiar tipo de entorno de ejecución*), la siguiente celda clona el repositorio para tener el corpus y `queries.jsonl`, e instala `sentence-transformers` y `chromadb`. En local no hace nada (allí se instalan con `requirements.txt`).

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !git clone --quiet https://github.com/kevinMGII/uc3m-pln-rag-2026.git
    %cd uc3m-pln-rag-2026
    %pip install --quiet sentence-transformers chromadb

/content/uc3m-pln-rag-2026
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


In [ ]:
import os
import re
import json
import hashlib
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import chromadb

os.environ.setdefault('HF_HUB_DISABLE_SYMLINKS_WARNING', '1')   # aviso inofensivo de Hugging Face en Windows
from transformers import AutoTokenizer
from transformers.utils import logging as hf_logging
from sentence_transformers import SentenceTransformer
hf_logging.set_verbosity_error()

pd.set_option('display.max_colwidth', 100)

CORPUS_ZIP = Path('corpus/corpus_misiones_espaciales_json.zip')
QUERIES_PATH = Path('queries.jsonl')
CACHE_DIR = Path('cache')                  # embeddings calculados (no se sube al repositorio)
CHROMA_DIR = Path('chroma_db')             # base de datos vectorial (no se sube al repositorio)

CONFIG = {
    # Fragmentación: los mismos valores que en hito1.ipynb
    'chunk_words': 300,
    'chunk_overlap': 45,
    'max_words_whole': 400,
    # Sentence embeddings
    'embedding_model': 'BAAI/bge-small-en-v1.5',
    'max_seq_length': 512,      # tokens que lee el modelo; el resto del fragmento se ignora
    'batch_size': 32,
    'normalize': True,          # vectores de norma 1: similitud coseno = producto escalar
    # Instrucción que bge-small-en-v1.5 recomienda añadir a las consultas (no a los fragmentos)
    'query_instruction': 'Represent this sentence for searching relevant passages: ',
}
N_FRAGMENTS_H1 = 19_620         # fragmentos del Hito 1, para comprobar que son los mismos

## **1. Corpus y fragmentos**

Mismo código que las secciones 1 y 2 de `hito1.ipynb`: se carga el corpus y se divide cada documento en fragmentos de 200–300 palabras con un solapamiento de 45. Los fragmentos no se guardan en un fichero: se regeneran aquí exactamente igual, y lo comprobamos con dos condiciones:

* salen los mismos **19.620 fragmentos**, y
* existen todos los fragmentos de origen (`source_chunk`) de las consultas de `queries.jsonl`.

A diferencia del Hito 1, guardamos también el **título, la URL y la fuente** de cada fragmento: los necesitaremos para que el chatbot cite de dónde saca la respuesta.

In [ ]:
with zipfile.ZipFile(CORPUS_ZIP) as z:
    index = json.loads(z.read('indice.json'))
    docs = pd.DataFrame([json.loads(z.read(row['file'])) for row in index])
docs = docs[['doc_id', 'title', 'url', 'source', 'text']]
assert docs.doc_id.is_unique and docs.text.str.strip().str.len().gt(0).all()

WORD_RE = re.compile(r"\b\w+(?:['’-]\w+)*\b")

def split_document(text):
    # Idéntica a la de hito1.ipynb
    spans = [m.span() for m in WORD_RE.finditer(text)]
    n = len(spans)
    if n <= CONFIG['max_words_whole']:
        return [text.strip()]
    step = CONFIG['chunk_words'] - CONFIG['chunk_overlap']
    k = int(np.ceil((n - CONFIG['chunk_overlap']) / step))
    size = (n + (k - 1) * CONFIG['chunk_overlap']) / k
    starts = [round(i * (size - CONFIG['chunk_overlap'])) for i in range(k)]
    ends = [round(s + size) for s in starts[:-1]] + [n]
    return [text[spans[s][0]:spans[e - 1][1]] for s, e in zip(starts, ends)]

rows = []
for doc in docs.itertuples():
    for i, chunk in enumerate(split_document(doc.text), start=1):
        rows.append({'chunk_id': f'{doc.doc_id}__{i:04d}', 'doc_id': doc.doc_id, 'title': doc.title,
                     'url': doc.url, 'source': doc.source, 'text': chunk})
frags = pd.DataFrame(rows)

queries = pd.read_json(QUERIES_PATH, lines=True, dtype={'qid': str})
assert len(frags) == N_FRAGMENTS_H1, f'Salen {len(frags)} fragmentos y en el Hito 1 había {N_FRAGMENTS_H1}'
assert set(queries.source_chunk) <= set(frags.chunk_id), 'Hay fragmentos de origen de las consultas que no existen'

print(f'{len(docs):,} documentos -> {len(frags):,} fragmentos (los mismos que en el Hito 1)')
print(f'{len(queries)} consultas en {QUERIES_PATH}; todos sus fragmentos de origen existen')
frags[['chunk_id', 'title', 'source', 'text']].head(3)

3,314 documentos -> 19,620 fragmentos (los mismos que en el Hito 1)
82 consultas en queries.jsonl; todos sus fragmentos de origen existen


,chunk_id,title,source,text
0,doc_0001__0001,AMC-6,Wikipedia 2000–2025,AMC-6\n\nNames | GE-6 (1997-2001) AMC-6 (2001-present) Rainbow 2 (2004-present)\n\nMission type ...
1,doc_0002__0001,AMC-8,Wikipedia 2000–2025,AMC-8\n\nNames | GE-8 (2000-2001) AMC-8 (2001-present) Aurora-3 (2000-present) Aurora-III\n\nMis...
2,doc_0003__0001,Astra 2B,Wikipedia 2000–2025,Astra 2B\n\nMission type | Communications\n\nOperator | SES\n\nCOSPAR ID | 2000-054A\n\nSATCAT n...


## **2. Paso 1: sentence embeddings**

Un **sentence embedding** es un vector denso que representa el significado de un texto completo (una frase o, en nuestro caso, un fragmento). Lo calcula un modelo *encoder* tipo BERT:

1. El texto se divide en *tokens* y el Transformer calcula un vector por token, que depende del contexto (`last_hidden_state`, como en la sección 1.1.3 del *Hugging Face Transformers Tutorial I*).
2. Esos vectores se combinan en uno solo: la media de todos (*mean pooling*) o el vector del token `[CLS]`.
3. A diferencia de promediar vectores de Word2Vec o GloVe (*Text Vectorization II*), estos modelos se han entrenado específicamente para que **textos con el mismo significado queden cerca**, aunque usen palabras distintas. Es lo que debería ayudar en consultas como *Guiana Space Centre* frente a *Kourou*, que fallaban en el Hito 1.

A diferencia de TF-IDF y BM25, el modelo lee el **texto original** del fragmento: no se lematiza ni se quitan stopwords, porque el modelo necesita las frases completas para entender el contexto.

Usamos la librería [`sentence-transformers`](https://www.sbert.net/), que carga modelos de Hugging Face y hace el *pooling* y la normalización.

### *2.1. Elección del modelo: cuánto texto puede leer*

Un modelo de embeddings solo lee un número máximo de *tokens*; si el texto es más largo, **corta el resto sin avisar**. Comparamos dos modelos pequeños de inglés (nuestro corpus está en inglés), del mismo tamaño (384 dimensiones):

* `sentence-transformers/all-MiniLM-L6-v2`: el más usado en tutoriales, lee 256 *tokens*.
* `BAAI/bge-small-en-v1.5`: entrenado específicamente para recuperación, lee 512 *tokens*.

Medimos la longitud de nuestros fragmentos en *tokens* (los dos modelos usan el mismo tokenizador de BERT):

In [ ]:
CANDIDATES = {'sentence-transformers/all-MiniLM-L6-v2': 256, 'BAAI/bge-small-en-v1.5': 512}

lengths = {}
for name in CANDIDATES:
    tokenizer = AutoTokenizer.from_pretrained(name)
    lengths[name] = np.array([len(ids) for ids in tokenizer(frags.text.tolist())['input_ids']])

pd.DataFrame([{'modelo': name, 'tokens que lee': limit,
               'mediana de tokens por fragmento': int(np.median(lengths[name])),
               'percentil 90': int(np.percentile(lengths[name], 90)),
               '% fragmentos cortados': round(100 * (lengths[name] > limit).mean(), 1),
               '% de tokens perdidos': round(100 * np.maximum(lengths[name] - limit, 0).sum() / lengths[name].sum(), 1)}
              for name, limit in CANDIDATES.items()]).set_index('modelo')

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

,tokens que lee,mediana de tokens por fragmento,percentil 90,% fragmentos cortados,% de tokens perdidos
modelo,,,,,
sentence-transformers/all-MiniLM-L6-v2,256,385,521,99.4,39.2
BAAI/bge-small-en-v1.5,512,385,521,10.6,5.3


**Decisión: `BAAI/bge-small-en-v1.5`.** Nuestros fragmentos tienen una mediana de unos 385 *tokens*, así que `all-MiniLM-L6-v2` cortaría casi todos y no vería aproximadamente el último tercio de cada uno. `bge-small-en-v1.5` lee los fragmentos completos en casi el 90 % de los casos, y en el resto solo pierde el final.

Más adelante (paso 3) podremos comparar otros modelos midiendo la recuperación con las consultas del Hito 1; de momento elegimos el que mejor encaja con el tamaño de nuestros fragmentos.

Un detalle de este modelo para el paso 3: en búsqueda, recomienda añadir a las **consultas** (no a los fragmentos) la instrucción `"Represent this sentence for searching relevant passages: "`.

In [ ]:
model = SentenceTransformer(CONFIG['embedding_model'])      # usa la GPU si la hay
model.max_seq_length = CONFIG['max_seq_length']

print('Modelo:', CONFIG['embedding_model'], '· dispositivo:', model.device)
print('Tokens máximos:', model.max_seq_length)
# El método cambió de nombre en las versiones recientes de sentence-transformers (Colab usa una más nueva)
get_dim = getattr(model, 'get_embedding_dimension', None) or model.get_sentence_embedding_dimension
embedding_dim = get_dim()
print('Dimensión de los embeddings:', embedding_dim)
print(model)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo: BAAI/bge-small-en-v1.5 · dispositivo: cuda:0
Tokens máximos: 512
Dimensión de los embeddings: 384
SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({})
)


### *2.2. Cálculo de los embeddings*

Calculamos un vector por fragmento. Es la parte lenta del hito (unos minutos con GPU; bastante más en CPU), así que se hace **una sola vez** y se guarda en `cache/` (que no se sube al repositorio, como la caché de lemas del Hito 1). El nombre del fichero depende del modelo y de los fragmentos: si cambia cualquiera de los dos, se recalcula.

Pedimos los vectores **normalizados** (norma 1): así la similitud coseno entre dos fragmentos es simplemente su producto escalar.

In [ ]:
%%time
emb_key = hashlib.sha256(json.dumps(
    [CONFIG['embedding_model'], CONFIG['max_seq_length'], CONFIG['normalize'],
     frags.chunk_id.tolist(), frags.text.tolist()]).encode()).hexdigest()
emb_file = CACHE_DIR / f'embeddings_{emb_key[:16]}.npy'

if emb_file.exists():
    embeddings = np.load(emb_file)
    print('Embeddings cargados de', emb_file)
else:
    embeddings = model.encode(frags.text.tolist(), batch_size=CONFIG['batch_size'],
                              normalize_embeddings=CONFIG['normalize'], convert_to_numpy=True,
                              show_progress_bar=False).astype(np.float32)
    CACHE_DIR.mkdir(exist_ok=True)
    np.save(emb_file, embeddings)
    print('Embeddings calculados y guardados en', emb_file)

Embeddings calculados y guardados en cache/embeddings_7191591d45aa8aab.npy
CPU times: user 3min 38s, sys: 478 ms, total: 3min 39s
Wall time: 3min 31s


### *2.3. Comprobaciones*

Una fila por fragmento, en el mismo orden que `frags`, sin valores vacíos y con norma 1:

In [ ]:
norms = np.linalg.norm(embeddings, axis=1)
assert embeddings.shape == (len(frags), embedding_dim)
assert np.isfinite(embeddings).all()
assert np.allclose(norms, 1, atol=1e-3)

print(f'Matriz de embeddings: {embeddings.shape[0]:,} fragmentos x {embeddings.shape[1]} dimensiones '
      f'({embeddings.nbytes / 1e6:.0f} MB en float32)')
print(f'Normas: entre {norms.min():.4f} y {norms.max():.4f}')
print('Primeros valores del primer fragmento:', embeddings[0, :8].round(3))

Matriz de embeddings: 19,620 fragmentos x 384 dimensiones (30 MB en float32)
Normas: entre 1.0000 y 1.0000
Primeros valores del primer fragmento: [-0.004  0.043  0.012 -0.058  0.001 -0.03  -0.032  0.068]


### *2.4. Primer vistazo: ¿los fragmentos parecidos quedan cerca?*

Antes de evaluar la recuperación con las consultas (paso 3), comprobamos que los vectores tienen sentido.

**Similitud entre fragmentos.** Si el modelo funciona, dos fragmentos del **mismo documento** deberían parecerse más que dos fragmentos al azar:

In [ ]:
rng = np.random.default_rng(2026)
n_pairs = 5000

# Pares de fragmentos consecutivos del mismo documento
same_doc = np.flatnonzero(frags.doc_id.values[1:] == frags.doc_id.values[:-1])
same_doc = rng.choice(same_doc, size=min(n_pairs, len(same_doc)), replace=False)
sim_same = np.einsum('ij,ij->i', embeddings[same_doc], embeddings[same_doc + 1])

# Pares de fragmentos al azar de documentos distintos
a, b = rng.integers(len(frags), size=(2, n_pairs))
different = frags.doc_id.values[a] != frags.doc_id.values[b]
sim_random = np.einsum('ij,ij->i', embeddings[a[different]], embeddings[b[different]])

pd.DataFrame({'mismo documento (consecutivos)': pd.Series(sim_same).describe(),
              'documentos distintos (al azar)': pd.Series(sim_random).describe()}).T[['count', 'mean', '25%', '50%', '75%']].round(3)

,count,mean,25%,50%,75%
mismo documento (consecutivos),5000.0,0.827,0.793,0.834,0.869
documentos distintos (al azar),4960.0,0.612,0.574,0.617,0.657


**Vecinos más cercanos.** Para un fragmento cualquiera, buscamos los fragmentos con mayor similitud coseno (producto escalar). Es una búsqueda por fuerza bruta sobre toda la matriz, solo para inspeccionar; en el paso 2 esta búsqueda la hará la base de datos vectorial.

In [ ]:
def nearest_fragments(vector, k=5, exclude=None):
    sims = embeddings @ vector
    order = [i for i in np.argsort(-sims) if i != exclude][:k]
    return frags.loc[order, ['chunk_id', 'title', 'source']].assign(similitud=sims[order].round(3),
                                                                     texto=frags.text[order].str[:120].str.replace(r'\s+', ' ', regex=True))

example_id = queries.source_chunk.iloc[0]               # el fragmento de origen de la primera consulta
i = frags.index[frags.chunk_id == example_id][0]
print(f'Fragmento: {example_id} · {frags.title[i]}')
print(' '.join(frags.text[i].split())[:400], '...\n')
nearest_fragments(embeddings[i], exclude=i)

Fragmento: doc_0488__0002 · Eutelsat 33E
lb) and is expected to operate for 15 years. The spacecraft has 64 Ku-band transponders , broadcasting satellite television and radio to Europe, North Africa and the Middle East. Hot Bird 10, as it was then named, was launched by Arianespace using an Ariane 5ECA carrier rocket, as part of a dual launch with the NSS-9 spacecraft. Two French military satellites, Spirale-A and Spirale-B were also car ...



,chunk_id,title,source,similitud,texto
9866,doc_3066__0003,Eutelsat 16B,Wikipedia 1957–1999,0.890,stream which could be received in the downlink by standard IRDs. Launch The satellite was launch...
1441,doc_0488__0001,Eutelsat 33E,Wikipedia 2000–2025,0.877,Eutelat 33E Names | Hot Bird 10 (2009) Atlantic Bird 4A (2009–2011) Eutelsat 3C (2011–2013) Hot ...
324,doc_0115__0002,Eutelsat 8 West C,Wikipedia 2000–2025,0.839,"Alcatel Space based on the Spacebus-3000B3 satellite bus , Hot Bird 6 is a 3,905 kg (8,609 lb) s..."
172,doc_0065__0001,Eutelsat 12 West B,Wikipedia 2000–2025,0.831,Eutelsat 12 West B Names | Atlantic Bird 2 (2001–2012) Eutelsat 8 West A (2012–2015) Mission typ...
813,doc_0301__0001,Hot Bird 13B,Wikipedia 2000–2025,0.829,Hot Bird 13B Names | Hot Bird 8 (2006–2012) Hot Bird 13B (2012–2023) Eutelsat Hot Bird 13B Missi...


**Resumen del paso 1.** Los 19.620 fragmentos del Hito 1 tienen ya su *sentence embedding* (matriz de 19.620 × 384, normalizada) calculado con `BAAI/bge-small-en-v1.5`. Los fragmentos del mismo documento quedan bastante más cerca que los de documentos distintos.

## **3. Paso 2: base de datos vectorial**

El enunciado exige guardar los embeddings en una **base de datos vectorial** (sección 5.4), que es la que, ante una consulta, devuelve los fragmentos más cercanos. Cita como ejemplos **FAISS** y **Chroma** (ColBERT no es una base de datos, sino un modelo de recuperación).

Usamos **Chroma**, porque guarda **juntos**, por cada fragmento:

| Qué | Para qué |
|---|---|
| el **vector** (embedding) | buscar por similitud con la consulta |
| el **texto** del fragmento | pasárselo al LLM como contexto (generación) |
| los **metadatos**: `doc_id`, título, URL, fuente | ranking por documento y **citar las fuentes**, uno de los requisitos funcionales |

Con FAISS habría que guardar el texto y los metadatos por separado. Además, Chroma guarda la base en disco (`chroma_db/`), así que solo se construye una vez.

Detalles de la configuración:

* **Distancia coseno** (`space: cosine`), la misma similitud que usa el modelo de embeddings. Chroma devuelve la *distancia* coseno, $1 - \text{similitud}$.
* No le damos a Chroma ningún modelo (`embedding_function=None`): le pasamos los vectores ya calculados en el paso 1, y en las consultas calcularemos nosotros su vector con el mismo modelo.
* El nombre de la colección incluye el mismo código que la caché de embeddings: si cambian el modelo o los fragmentos, se crea una colección nueva en vez de mezclar vectores incompatibles.

In [ ]:
%%time
client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection_name = f'fragmentos_{emb_key[:16]}'
existing = {c.name if hasattr(c, 'name') else c for c in client.list_collections()}

if collection_name in existing and client.get_collection(collection_name).count() == len(frags):
    collection = client.get_collection(collection_name)
    print(f'Colección "{collection_name}" cargada de {CHROMA_DIR}/')
else:
    if collection_name in existing:              # a medio construir: se rehace
        client.delete_collection(collection_name)
    collection = client.create_collection(collection_name, configuration={'hnsw': {'space': 'cosine'}},
                                          embedding_function=None)
    batch_size = client.get_max_batch_size()     # Chroma limita cuántos elementos se añaden de golpe
    for start in range(0, len(frags), batch_size):
        part = frags.iloc[start:start + batch_size]
        collection.add(ids=part.chunk_id.tolist(),
                       embeddings=embeddings[start:start + batch_size],
                       documents=part.text.tolist(),
                       metadatas=part[['doc_id', 'title', 'url', 'source']].to_dict('records'))
    print(f'Colección "{collection_name}" creada en {CHROMA_DIR}/')

print(f'{collection.count():,} fragmentos en la base de datos vectorial')

Colección "fragmentos_7191591d45aa8aab" creada en chroma_db/
19,620 fragmentos en la base de datos vectorial
CPU times: user 33 s, sys: 3.94 s, total: 37 s
Wall time: 36 s


### *3.1. Comprobaciones*

**Contenido.** La base de datos tiene un elemento por fragmento, y para unos cuantos al azar comprobamos que el vector, el texto y los metadatos guardados son exactamente los nuestros:

In [ ]:
assert collection.count() == len(frags)

sample = frags.sample(100, random_state=2026)
stored = collection.get(ids=sample.chunk_id.tolist(), include=['embeddings', 'documents', 'metadatas'])
stored_by_id = {cid: (emb, doc, meta) for cid, emb, doc, meta in
                zip(stored['ids'], stored['embeddings'], stored['documents'], stored['metadatas'])}
for row in sample.itertuples():
    emb, doc, meta = stored_by_id[row.chunk_id]
    assert np.allclose(emb, embeddings[row.Index], atol=1e-6)
    assert doc == row.text
    assert meta == {'doc_id': row.doc_id, 'title': row.title, 'url': row.url, 'source': row.source}
print('OK: vector, texto y metadatos coinciden en 100 fragmentos al azar\n')

example = sample.iloc[0]
emb, doc, meta = stored_by_id[example.chunk_id]
print('Ejemplo de lo que guarda Chroma para', example.chunk_id)
print('  metadatos:', meta)
print('  vector:   ', np.round(emb[:6], 3), '...', f'({len(emb)} valores)')
print('  texto:    ', ' '.join(doc.split())[:200], '...')

OK: vector, texto y metadatos coinciden en 100 fragmentos al azar

Ejemplo de lo que guarda Chroma para doc_1823__0001
  metadatos: {'title': 'SOLRAD 7A', 'url': 'https://en.wikipedia.org/wiki/SOLRAD_7A', 'doc_id': 'doc_1823', 'source': 'Wikipedia 1957–1999'}
  vector:    [-0.065  0.044  0.027  0.025  0.057  0.043] ... (384 valores)
  texto:     SOLRAD 7A Mission type | Solar X-Ray Operator | NRL COSPAR ID | 1964-001D SATCAT no. | 00730 Spacecraft properties Launch mass | 45.4 kilograms (100 lb) Start of mission Launch date | January 11, 1964 ...


**Búsqueda aproximada frente a exacta.** Para ser rápida, Chroma no compara la consulta con los 19.620 vectores, sino que usa un índice aproximado (**HNSW**, un grafo de vecinos). Comprobamos que sus resultados coinciden con la búsqueda exacta por fuerza bruta del paso 1: para 200 fragmentos al azar usados como consulta, qué parte de los 10 vecinos exactos devuelve Chroma.

In [ ]:
probe = np.random.default_rng(2026).choice(len(frags), size=200, replace=False)
result = collection.query(query_embeddings=embeddings[probe], n_results=10)
exact = np.argsort(-(embeddings[probe] @ embeddings.T), axis=1)[:, :10]

overlap = [len(set(ids) & set(frags.chunk_id.values[e])) / 10 for ids, e in zip(result['ids'], exact)]
print(f'Coincidencia media del top 10 de Chroma con la búsqueda exacta: {np.mean(overlap):.1%}')
print(f'Consultas con el top 10 completo: {np.mean(np.array(overlap) == 1):.1%}')

Coincidencia media del top 10 de Chroma con la búsqueda exacta: 100.0%
Consultas con el top 10 completo: 99.5%


### *3.2. Primera búsqueda con una consulta*

Así se usará la base de datos con una pregunta: se calcula el vector de la consulta con el **mismo modelo** (añadiendo la instrucción que recomienda `bge-small-en-v1.5` para las consultas) y se piden a Chroma los fragmentos más cercanos, con su texto y sus metadatos.

Es solo un ejemplo para ver que funciona. La **evaluación** con las 82 consultas (Recall@5, Recall@10 y MRR@10 a nivel de documento, comparada con TF-IDF y BM25) es el paso 3.

In [ ]:
def search(query, k=5):
    # Devuelve los k fragmentos más cercanos a la consulta, con su texto y metadatos
    q_emb = model.encode([CONFIG['query_instruction'] + query], normalize_embeddings=CONFIG['normalize'])
    r = collection.query(query_embeddings=q_emb, n_results=k, include=['documents', 'metadatas', 'distances'])
    return pd.DataFrame({'chunk_id': r['ids'][0],
                         'doc_id': [m['doc_id'] for m in r['metadatas'][0]],
                         'title': [m['title'] for m in r['metadatas'][0]],
                         'similitud': [round(1 - d, 3) for d in r['distances'][0]],
                         'texto': [' '.join(t.split())[:120] for t in r['documents'][0]]},
                        index=range(1, k + 1))

q = queries.iloc[0]
relevant_doc = q.relevant_ids[0]
print(f'Consulta {q.qid}: {q.query}')
print(f'Documento relevante: {relevant_doc} · {docs.set_index("doc_id").title[relevant_doc]}\n')
results = search(q.query)
results.style.apply(lambda row: ['background-color: #d4edda' if row.doc_id == relevant_doc else '' for _ in row], axis=1)

Consulta q01: French satellite carrying military payloads launched from Guiana Space Centre on February 12, 2009 with Ariane rocket
Documento relevante: doc_0488 · Eutelsat 33E



,chunk_id,doc_id,title,similitud,texto
1,doc_1580__0002,doc_1580,Sentinel-1D,0.769000,"Sentinel-1 satellites can detect ground movement changes in terms of millimetres per year, which is used for monitoring"
2,doc_0891__0002,doc_0891,Athena-Fidus,0.758000,"Thales Alenia Space . The satellite bus is a Spacebus 4000B2 , manufactured in Cannes Mandelieu Space Center . Launch Th"
3,doc_3219__0011,doc_3219,Topex/Poseidon (336KB),0.749000,inches (10 cm) or better. The GPS receiver operates at 1227.6 MHz and 1575.4 MHz. MISSION PHASES The French-provided Ari
4,doc_3220__0011,doc_3220,(8.2M),0.749000,inches (10 cm) or better. The GPS receiver operates at 1227.6 MHz and 1575.4 MHz. MISSION PHASES The French-provided Ari
5,doc_2917__0001,doc_2917,Cerise (satellite),0.748000,CERISE Illustration of debris colliding with Cerise Mission type | Military reconnaissance COSPAR ID | 1995-033B SATCAT


## **Resumen**

* **Paso 1:** los 19.620 fragmentos del Hito 1 tienen su *sentence embedding* (`BAAI/bge-small-en-v1.5`, 384 dimensiones, normalizados), guardado en `cache/`.
* **Paso 2:** los vectores están en una **base de datos vectorial Chroma** (`chroma_db/`), junto con el texto y los metadatos de cada fragmento (`doc_id`, título, URL, fuente), con distancia coseno. Su búsqueda aproximada coincide con la exacta, y ya puede responder consultas.

**Siguiente paso (3):** recuperación densa con las 82 consultas de `queries.jsonl`, con el mismo protocolo del Hito 1 (cada documento con la puntuación de su mejor fragmento, sin repetidos, Recall@5, Recall@10 y MRR@10), y comparación con TF-IDF y BM25.